In [0]:
resource_path = '/Volumes/workspace/pj_sales/sales_resources/origin/'
processed_path = '/Volumes/workspace/pj_sales/sales_resources/processed/'

In [0]:
df_sales = spark.read\
    .format('csv')\
    .option('sep', ',')\
    .option('inferSchema', 'true')\
    .option('header', 'true')\
    .load(resource_path)

In [0]:
from pyspark.sql.functions import current_timestamp, col, from_utc_timestamp
df_sales = df_sales\
    .withColumn('ingestion_timestamp', from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo'))\
    .withColumn('source_file', col('_metadata.file_path'))

In [0]:
df_sales.write\
    .format('delta')\
    .mode('append')\
    .option('mergeSchema', 'true')\
    .saveAsTable('workspace.pj_sales.tb_sales_bronze')

In [0]:
processed_files = [f for f in dbutils.fs.ls(resource_path) if f.name.endswith('.csv')]
for file in processed_files:
  dbutils.fs.mv(file.path, processed_path + file.name)
  print(f'File {file.name} moved to processed.')
print('All files processed!')